# SpecTexNet: full experiment run on Colab / Kaggle

Runtime > Change runtime type > **GPU** (T4 is enough; A100/L4 is faster).

Every step is resumable. If the session dies, rerun all cells: finished folds are skipped. Results are copied to Google Drive after each experiment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/FabricDefect_v2'
!mkdir -p $DRIVE_OUT

In [ ]:
# 1) get the code (replace with your GitHub URL)
!git clone https://github.com/CodeWithYeasin/FabricDefectDetection.git /content/FabricDefectDetection
%cd /content/FabricDefectDetection
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) dataset: upload kaggle.json once to Drive root, or use files.upload()
!mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nexuswho/fabric-defects-dataset -p data/raw -q
!unzip -q -o data/raw/fabric-defects-dataset.zip -d data/raw/
!ls data/raw

In [ ]:
# 3) restore earlier results from Drive (resume support)
!mkdir -p results data && cp -rn $DRIVE_OUT/results/* results/ 2>/dev/null; cp -n $DRIVE_OUT/data/* data/ 2>/dev/null; ls results

In [ ]:
# 4) index + de-duplicate + folds  (look at data/dedup_report.json afterwards)
!python scripts/prepare_data.py --roots "data/raw/Fabric Defects Dataset" --out data
!cat data/dedup_report.json | head -40
!cp data/splits.csv data/dedup_report.json data/duplicate_pairs.csv $DRIVE_OUT/ 2>/dev/null; mkdir -p $DRIVE_OUT/data && cp data/*.csv data/*.json $DRIVE_OUT/data/

In [ ]:
# 5) v1 baseline on the new folds
!python scripts/run_legacy_v1.py
!rsync -a results/ $DRIVE_OUT/results/

In [ ]:
# 6) all deep models (about 10-14 h on a T4 in total; run over several sessions if needed)
EXPS = ['spectex_convnext_tiny','spectex_resnet50','spectex_efficientnet_b0',
        'resnet50','efficientnet_b0','mobilenetv3_large_100','convnext_tiny','swin_tiny','deit_small',
        'abl_no_spectral','abl_concat_fusion','abl_fixed_sr','abl_single_scale',
        'abl_aug_flip','abl_aug_naive_rot','abl_loss_ce']
for e in EXPS:
    !python scripts/train.py --exp {e} --skip-done
    !rsync -a results/ $DRIVE_OUT/results/

In [ ]:
# 7) efficiency, figures, statistics, LaTeX tables
!python scripts/benchmark_efficiency.py
!python scripts/figure_architecture.py
!python scripts/figure_method.py
!python scripts/explain.py --exp spectex_convnext_tiny --fold 0 --per-class 2
!python scripts/aggregate.py --proposed spectex_convnext_tiny
!rsync -a results/ $DRIVE_OUT/results/ && rsync -a paper/ $DRIVE_OUT/paper/

In [ ]:
# 8) deployment model on all data + export to the HF Space folder
!python scripts/train.py --exp spectex_convnext_tiny --full
!python scripts/export_model.py --run results/spectex_convnext_tiny/full
!rsync -a deploy/ $DRIVE_OUT/deploy/

In [ ]:
import json; s=json.load(open('results/summary.json'))
for k,v in sorted(s.items(), key=lambda kv: -kv[1]['cv']['macro_f1']['mean']):
    print(f"{k:32s} macroF1 {100*v['cv']['macro_f1']['mean']:.2f} ± {100*v['cv']['macro_f1']['std']:.2f}   acc {100*v['cv']['accuracy']['mean']:.2f}")